# JAX Research Interview Practice Notebook

Exercises designed to prepare you for research coding interviews involving JAX and numerical computing.

**Difficulty Levels:**
- 🟢 **Basic** - Core concepts and operations
- 🟡 **Intermediate** - Combining multiple concepts
- 🔴 **Advanced** - Algorithm implementation & optimization
- ⚫ **Research/Interview Level** - Complex algorithms, edge cases, performance tuning

**Interview Strategy Tips:**
1. Understand the math behind the algorithm first
2. Write for correctness before optimization
3. Know when to use `jit`, `grad`, `vmap`
4. Test on small inputs before scaling
5. Ask about constraints: shape, dtype, performance requirements

---

## Setup: Imports and Utilities

Run this cell first to set up everything you need.

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap
from jax import random
import numpy as np
import time

# Utility function to time code
def time_it(fn, *args, n_runs=100, **kwargs):
    """Time a function over n_runs"""
    start = time.time()
    for _ in range(n_runs):
        _ = fn(*args, **kwargs)
    return (time.time() - start) / n_runs

# Utility for checking answers
def check_close(actual, expected, rtol=1e-5, atol=1e-5):
    """Check if arrays are close"""
    if isinstance(actual, (list, tuple)):
        return all(jnp.allclose(a, e, rtol=rtol, atol=atol) for a, e in zip(actual, expected))
    return jnp.allclose(actual, expected, rtol=rtol, atol=atol)

print("✓ Setup complete")

# Section 1: Basic Operations & Immutability 🟢

**Context**: Interviews often start by checking if you understand JAX fundamentals, especially immutability.

### Exercise 1.1: Array Update Pattern 🟢

**Problem**: In NumPy, you'd update arrays with `arr[i] = value`. JAX arrays are immutable.  
Complete the function to update multiple indices efficiently.

**Prompt**: Fill in the missing code

In [ ]:
def update_indices(arr, indices, values):
    """
    Update multiple positions in a JAX array.
    
    Args:
        arr: JAX array to update
        indices: array of indices to update
        values: array of values to insert
        
    Returns:
        Updated array
        
    Example: 
        arr = [1, 2, 3, 4, 5]
        indices = [0, 2, 4]
        values = [10, 30, 50]
        Result: [10, 2, 30, 4, 50]
    """
    # FILL IN HERE
    pass

# Test
arr = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0])
indices = jnp.array([0, 2, 4])
values = jnp.array([10.0, 30.0, 50.0])
result = update_indices(arr, indices, values)
expected = jnp.array([10.0, 2.0, 30.0, 4.0, 50.0])

print(f"Result: {result}")
print(f"Expected: {expected}")
print(f"✓ PASS" if check_close(result, expected) else "✗ FAIL")

### Exercise 1.2: Element-wise Operations 🟢

**Problem**: Implement a clipped normalization (normalize to [min_val, max_val] range).

**Hint**: Use JAX operations for maximum efficiency.

In [ ]:
def normalize_clipped(x, min_val=0.0, max_val=1.0):
    """
    Normalize array to [min_val, max_val] range by clipping after min-max scaling.
    
    Formula:
    1. x_norm = (x - min(x)) / (max(x) - min(x))  # Scale to [0, 1]
    2. x_scaled = x_norm * (max_val - min_val) + min_val  # Scale to [min_val, max_val]
    3. Clip to [min_val, max_val]
    
    Returns:
        Normalized array in range [min_val, max_val]
    """
    # FILL IN HERE
    pass

# Test
x = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0])
result = normalize_clipped(x, min_val=-1.0, max_val=1.0)
expected = jnp.array([-1.0, -0.5, 0.0, 0.5, 1.0])

print(f"Result: {result}")
print(f"Expected: {expected}")
print(f"✓ PASS" if check_close(result, expected) else "✗ FAIL")

# Section 2: Automatic Differentiation & Optimization 🟡

**Context**: Gradient computation is fundamental. Many interviews focus on implementing gradient-based algorithms.

### Exercise 2.1: Numerical Gradient Checking 🟡

**Problem**: Implement numerical gradient checking to verify analytical gradients.

**Interview Tip**: Often interviewers ask "How do you verify your gradient implementation?" This is the answer.

**Prompt**: Complete the finite difference calculation.

In [ ]:
def numerical_gradient(f, x, eps=1e-5):
    """
    Compute numerical gradient using central difference.
    
    Formula: df/dx ≈ (f(x+ε) - f(x-ε)) / (2ε)
    
    Args:
        f: Function to differentiate
        x: Point at which to compute gradient
        eps: Small difference for numerical approximation
        
    Returns:
        Numerical gradient
    """
    # FILL IN HERE
    pass

def verify_gradient(f, x, eps=1e-5, rtol=1e-4):
    """Verify that analytical gradient matches numerical gradient"""
    numerical = numerical_gradient(f, x, eps=eps)
    analytical = grad(f)(x)
    
    matches = check_close(numerical, analytical, rtol=rtol)
    print(f"Numerical:  {numerical}")
    print(f"Analytical: {analytical}")
    print(f"Match: {'✓ YES' if matches else '✗ NO'}")
    return matches

# Test
def f(x):
    return jnp.sum(x**3 + 2*x**2 - x)

x = jnp.array([1.0, 2.0, 3.0])
verify_gradient(f, x)

### Exercise 2.2: Gradient Descent Optimizer 🟡

**Problem**: Implement batch gradient descent.

**Key Points**:
- Learning rate control
- Convergence criteria
- Efficient updates with JAX

**Prompt**: Fill in the update rule.

In [ ]:
def gradient_descent(loss_fn, x0, learning_rate=0.01, n_steps=100):
    """
    Batch gradient descent optimizer.
    
    Args:
        loss_fn: Function that takes params and returns scalar loss
        x0: Initial parameters
        learning_rate: Step size
        n_steps: Number of iterations
        
    Returns:
        Final parameters, list of losses
    """
    x = x0
    losses = []
    grad_fn = grad(loss_fn)
    
    for step in range(n_steps):
        loss = loss_fn(x)
        g = grad_fn(x)
        
        # FILL IN UPDATE RULE
        # x = ...
        
        losses.append(loss)
        
    return x, jnp.array(losses)

# Test on quadratic function
def quadratic_loss(x):
    return jnp.sum((x - 3.0)**2)

x_init = jnp.array([0.0, 0.0, 0.0])
x_final, losses = gradient_descent(quadratic_loss, x_init, learning_rate=0.1, n_steps=50)

print(f"Initial x: {x_init}")
print(f"Final x: {x_final}")
print(f"Target x: [3.0, 3.0, 3.0]")
print(f"Initial loss: {losses[0]:.4f}, Final loss: {losses[-1]:.6f}")
print(f"Converged: {'✓ YES' if jnp.allclose(x_final, 3.0, atol=0.01) else '✗ NO'}")

# Section 3: Vectorization with vmap 🟡

**Context**: vmap is one of JAX's superpowers. It's essential for efficient batch operations in sampling and neural networks.

### Exercise 3.1: Vectorizing Gaussian Log Likelihood 🟡

**Problem**: Implement batch Gaussian log likelihood.

**Pattern**: Write it for a single sample first, then vectorize with vmap.

**Prompt**: Complete the single-sample function, then use vmap to vectorize it.

In [ ]:
def gaussian_log_likelihood_single(x, mean, cov_diag):
    """
    Compute log likelihood of single Gaussian sample.
    
    Formula: log p(x) = -0.5 * (d*log(2π) + sum(log(cov_diag)) + (x-mean)^T * cov_diag^-1 * (x-mean))
    
    Assumes diagonal covariance matrix (cov_diag contains diagonal elements).
    """
    # FILL IN HERE
    pass

def gaussian_log_likelihood_batch(batch_x, mean, cov_diag):
    """
    Vectorized version using vmap.
    
    Args:
        batch_x: Shape (batch_size, d)
        mean: Shape (d,)
        cov_diag: Shape (d,)
        
    Returns:
        Log likelihoods: Shape (batch_size,)
    """
    # FILL IN HERE (use vmap)
    pass

# Test
key = random.PRNGKey(0)
mean = jnp.array([0.0, 0.0])
cov_diag = jnp.array([1.0, 1.0])

# Single sample
x_single = jnp.array([1.0, 0.5])
log_prob_single = gaussian_log_likelihood_single(x_single, mean, cov_diag)

# Batch
batch_x = random.normal(key, shape=(100, 2))
log_prob_batch = gaussian_log_likelihood_batch(batch_x, mean, cov_diag)

print(f"Single log prob: {log_prob_single:.4f}")
print(f"Batch log probs shape: {log_prob_batch.shape}")
print(f"Batch first 3: {log_prob_batch[:3]}")

# Verify: first sample should match single computation
print(f"Match: {'✓ YES' if jnp.allclose(log_prob_batch[0], gaussian_log_likelihood_single(batch_x[0], mean, cov_diag)) else '✗ NO'}")

# Section 4: JIT Compilation & Performance 🟡

**Context**: Understanding when/how to use JIT is critical for interview performance discussions.

### Exercise 4.1: Performance - JIT vs Non-JIT 🟡

**Problem**: Compare performance of compiled vs non-compiled functions.

**Interview Question**: "When should you use JIT? What's the tradeoff?"

**Prompt**: Implement efficient MCMC step and measure speed difference.

In [ ]:
def log_prob(x):
    """Simple Gaussian log probability"""
    return -0.5 * jnp.sum(x**2)

def metropolis_step(key, x, proposal_std=1.0):
    """
    Single Metropolis-Hastings step.
    
    Returns: new_x, key, accept_prob
    
    Algorithm:
    1. Propose: x_new = x + proposal_std * z, z ~ N(0,1)
    2. Compute log acceptance ratio: log_alpha = log_prob(x_new) - log_prob(x)
    3. Accept with probability min(1, exp(log_alpha))
    """
    # FILL IN HERE
    pass

# Time comparison
key = random.PRNGKey(0)
x = jnp.array([0.0, 0.0, 0.0])

# Compiled version
metropolis_jit = jit(metropolis_step)

# Warmup JIT
_ = metropolis_jit(key, x)

# Benchmark
n_steps = 1000
print("Timing Metropolis-Hastings step:")

# Non-JIT
start = time.time()
for _ in range(n_steps):
    key, subkey = random.split(key)
    x, _, _ = metropolis_step(subkey, x)
t_no_jit = time.time() - start

# JIT
key = random.PRNGKey(0)
x = jnp.array([0.0, 0.0, 0.0])
start = time.time()
for _ in range(n_steps):
    key, subkey = random.split(key)
    x, _, _ = metropolis_jit(subkey, x)
t_jit = time.time() - start

print(f"Non-JIT: {t_no_jit:.4f}s ({n_steps/t_no_jit:.0f} steps/sec)")
print(f"JIT:     {t_jit:.4f}s ({n_steps/t_jit:.0f} steps/sec)")
print(f"Speedup: {t_no_jit/t_jit:.1f}x")

# Section 5: Advanced Algorithms 🔴

**Context**: These problems test deep understanding of JAX and numerical algorithms.

### Exercise 5.1: Hamiltonian Dynamics 🔴

**Problem**: Implement the leapfrog integrator for Hamiltonian dynamics.

**Used in**: HMC sampling, which is critical for Bayesian inference.

**Algorithm**:
```
p_half = p - (eps/2) * ∇q V(q)
q_new = q + eps * M^-1 * p_half
p_new = p_half - (eps/2) * ∇q V(q_new)
```

Where:
- V(q) is the potential energy (negative log probability)
- M is the mass matrix (usually identity)
- eps is the step size

**Prompt**: Fill in the leapfrog integrator.

In [ ]:
def potential_energy(q, grad_fn):
    """Compute potential energy V(q) = -log p(q)"""
    return -grad_fn(q)

def kinetic_energy(p):
    """Kinetic energy K(p) = 0.5 * p^T * p (with identity mass matrix)"""
    return 0.5 * jnp.sum(p**2)

def leapfrog_step(q, p, log_prob_fn, step_size, n_steps=10):
    """
    Leapfrog integrator for Hamiltonian dynamics.
    
    Args:
        q: position (parameters)
        p: momentum
        log_prob_fn: function that computes log probability
        step_size: integration step size (eps)
        n_steps: number of leapfrog steps
        
    Returns:
        q_new, p_new: new position and momentum
    """
    grad_log_prob = grad(log_prob_fn)
    
    for _ in range(n_steps):
        # Half step for momentum
        # FILL IN: p = ...
        
        # Full step for position
        # FILL IN: q = ...
        
        # Half step for momentum
        # FILL IN: p = ...
    
    return q, p

# Test
def log_prob_test(q):
    """Target: 2D Gaussian"""
    return -0.5 * jnp.sum(q**2)

q = jnp.array([0.0, 0.0])
p = random.normal(random.PRNGKey(0), shape=(2,))

# Simulate
q_new, p_new = leapfrog_step(q, p, log_prob_test, step_size=0.1, n_steps=10)

# Check energy conservation (important property of exact integrators)
H_initial = -log_prob_test(q) + kinetic_energy(p)
H_final = -log_prob_test(q_new) + kinetic_energy(p_new)

print(f"Initial position: {q}, momentum: {p}")
print(f"Final position: {q_new}, momentum: {p_new}")
print(f"Initial Hamiltonian: {H_initial:.6f}")
print(f"Final Hamiltonian: {H_final:.6f}")
print(f"Energy error: {abs(H_final - H_initial):.6f}")
print(f"Good conservation: {'✓ YES' if abs(H_final - H_initial) < 0.01 else '✗ NO'}")

### Exercise 5.2: Matrix Inversion & Numerical Stability 🔴

**Problem**: Implement numerically stable sampling from a multivariate Gaussian.

**Key Challenge**: Never invert covariance matrices directly - use Cholesky decomposition.

**Common Interview Question**: "How do you sample from N(μ, Σ) efficiently?"

**Prompt**: Implement using Cholesky, then compare numerical stability.

In [ ]:
def sample_mvn_naive(key, mean, cov, n_samples):
    """
    NAIVE approach: Sample from N(μ, Σ) by inverting covariance.
    
    NOT recommended - numerically unstable!
    """
    # FILL IN (compute L = inv(Σ)^0.5, then sample z ~ N(0,I), return μ + L^T * z)
    pass

def sample_mvn_stable(key, mean, cov, n_samples):
    """
    STABLE approach: Use Cholesky decomposition.
    
    Better: Σ = L * L^T, then sample z ~ N(0,I), return μ + L * z
    """
    # FILL IN (compute L = cholesky(Σ), then sample z ~ N(0,I), return μ + L * z)
    pass

# Test with ill-conditioned covariance
def create_ill_conditioned_cov(d, condition_number=1e6):
    """Create Σ = Q * diag(λ) * Q^T where λ ranges widely"""
    key = random.PRNGKey(0)
    Q, _ = jnp.linalg.qr(random.normal(key, shape=(d, d)))
    lambdas = jnp.logspace(0, jnp.log10(condition_number), d)
    cov = Q @ jnp.diag(lambdas) @ Q.T
    return cov

mean = jnp.array([1.0, 2.0])
cov = create_ill_conditioned_cov(2, condition_number=1e4)

key = random.PRNGKey(42)
samples_stable = sample_mvn_stable(key, mean, cov, 1000)

print(f"Covariance condition number: {jnp.linalg.cond(cov):.2e}")
print(f"Sample mean: {jnp.mean(samples_stable, axis=0)}")
print(f"Expected mean: {mean}")
print(f"Sample cov:\n{jnp.cov(samples_stable.T)}")
print(f"Expected cov:\n{cov}")

# Section 6: Research Interview Problems ⚫

**Context**: These are actual research-level interview problems covering edge cases, optimization tricks, and algorithm design.

### Exercise 6.1: Batch MCMC Sampling with vmap ⚫

**Problem**: Implement efficient parallel MCMC chain sampling using vmap.

**Challenge**: Multiple independent chains running in parallel, vectorized over batch dimension.

**Interview Value**: Tests understanding of vmap, random key splitting, and sampling algorithms.

**Prompt**: Complete the parallel chain runner.

In [ ]:
def single_chain_step(carry, _):
    """
    Single MCMC step for one chain.
    
    carry: (x, key, accept_count)
    returns: new_carry, diagnostics
    
    Target: 2D Gaussian
    """
    x, key, accept_count = carry
    
    # Log probability
    def log_prob(z):
        return -0.5 * jnp.sum(z**2)
    
    # Propose new state with Gaussian walk
    key, subkey = random.split(key)
    proposal = x + 0.5 * random.normal(subkey, shape=x.shape)
    
    # Metropolis acceptance
    log_alpha = log_prob(proposal) - log_prob(x)
    key, subkey = random.split(key)
    accept = jnp.log(random.uniform(subkey)) < log_alpha
    
    # Update state
    x_new = jnp.where(accept, proposal, x)
    accept_count_new = accept_count + accept
    
    return (x_new, key, accept_count_new), (x_new, accept)

def parallel_mcmc_chains(keys, x0, n_steps):
    """
    Run multiple MCMC chains in parallel.
    
    Args:
        keys: Shape (n_chains,) - random keys for each chain
        x0: Shape (n_chains, d) - initial state for each chain
        n_steps: number of MCMC steps
        
    Returns:
        samples: Shape (n_steps, n_chains, d)
        accept_rates: Shape (n_chains,)
    """
    # Process single chain
    def run_chain(key, x_init):
        carry = (x_init, key, 0.0)
        carry_final, (samples, accepts) = jax.lax.scan(single_chain_step, carry, None, length=n_steps)
        x_final, key_final, accept_count = carry_final
        accept_rate = accept_count / n_steps
        return samples, accept_rate
    
    # FILL IN: Use vmap to run chains in parallel
    # samples, accept_rates = vmap(...)(keys, x0)
    pass

# Test
n_chains = 4
n_steps = 100
d = 2

# Initialize
key = random.PRNGKey(0)
keys = random.split(key, n_chains)
x0 = random.normal(key, shape=(n_chains, d))

# Run parallel chains
samples, accept_rates = parallel_mcmc_chains(keys, x0, n_steps)

print(f"Samples shape: {samples.shape}")  # Should be (100, 4, 2)
print(f"Accept rates: {accept_rates}")
print(f"Average accept rate: {jnp.mean(accept_rates):.3f} (ideal: ~0.65 for 2D)")

# Diagnostics
print(f"\nChain diagnostics:")
for i in range(n_chains):
    chain_mean = jnp.mean(samples[:, i, :], axis=0)
    chain_std = jnp.std(samples[:, i, :], axis=0)
    print(f"Chain {i}: mean={chain_mean}, std={chain_std}")

### Exercise 6.2: Composing Transformations - Gradient through MCMC ⚫

**Problem**: Compute gradients through stochastic estimators (e.g., MCMC samples).

**Advanced Concept**: "Stop-gradient" operations to control what is differentiable.

**Interview Question**: "Can you differentiate through sampling?" Answer: "Yes, with stop-gradient or pathwise derivatives."

**Prompt**: Compute log marginal likelihood using MCMC samples.

In [ ]:
def estimate_log_marginal_likelihood(params, observations, n_samples=100):
    """
    Estimate log p(observations) using importance sampling.
    
    p(obs) ≈ (1/N) * Σ_i p(obs | z_i) where z_i ~ p(z)
    
    Args:
        params: generative model parameters
        observations: observed data
        n_samples: number of importance samples
        
    Returns:
        Estimated log marginal likelihood
    """
    key = random.PRNGKey(0)
    
    def log_joint(z):
        """Log p(obs, z | params) = log p(obs | z, params) + log p(z)"""
        # Prior: p(z) = N(0, I)
        log_prior = -0.5 * jnp.sum(z**2)
        
        # Likelihood: p(obs | z) = N(obs; z, σ^2)
        sigma = 0.1
        log_likelihood = -0.5 * jnp.sum(((observations - z) / sigma)**2)
        
        return log_likelihood + log_prior
    
    # Sample from prior
    key, subkey = random.split(key)
    z_samples = random.normal(subkey, shape=(n_samples, len(observations)))
    
    # Compute log weights (no gradients through sampling)
    log_weights = jax.lax.map(lambda z: log_joint(jax.lax.stop_gradient(z)), z_samples)
    
    # Log-sum-exp trick for numerical stability
    # log Σ exp(w) = max(w) + log Σ exp(w - max(w))
    max_log_weight = jnp.max(log_weights)
    log_marginal = max_log_weight + jnp.log(jnp.mean(jnp.exp(log_weights - max_log_weight)))
    
    return log_marginal

# Compute gradient of log marginal likelihood w.r.t. observations
def estimate_log_marginal_with_grad(observations, n_samples=100):
    """Compute both log_marginal and gradient"""
    params = jnp.array([])  # Dummy params
    
    log_marginal = estimate_log_marginal_likelihood(params, observations, n_samples)
    return log_marginal

# Test
observations = jnp.array([1.0, 2.0, 3.0])
log_marginal = estimate_log_marginal_with_grad(observations)

print(f"Estimated log marginal likelihood: {log_marginal:.4f}")

# Compute gradient - note: this is tricky because of sampling!
# The gradient won't flow through the random samples (due to stop_gradient)
# This is correct behavior for importance sampling
try:
    grad_fn = grad(lambda obs: estimate_log_marginal_with_grad(obs))
    g = grad_fn(observations)
    print(f"Gradient w.r.t. observations: {g}")
except Exception as e:
    print(f"Note: {e}")
    print("This is expected - importance sampling estimator has zero gradient through samples.")

# Section 7: Tricky Questions & Edge Cases ⚫

**Context**: These are gotchas and subtle performance/correctness issues that come up in research code.

### Tricky Question 7.1: Random Number Generation Pitfall ⚫

**Problem**: What's wrong with this code?

```python
def sample_multiple_chains(n_chains, n_steps):
    key = random.PRNGKey(0)
    chains = []
    for i in range(n_chains):
        chain = random.normal(key, shape=(n_steps, 2))
        chains.append(chain)
    return jnp.array(chains)
```

**Answer**: You'd get identical chains! PRNG key must be split for each chain.

**Correct Pattern**: 
```python
key = random.PRNGKey(0)
for i in range(n_chains):
    key, subkey = random.split(key)  # ← CRITICAL
    chain = random.normal(subkey, shape=(n_steps, 2))
```

**Verify this below:**

In [ ]:
# WRONG: Identical chains
def sample_wrong(n_chains, n_steps):
    key = random.PRNGKey(0)
    chains = []
    for i in range(n_chains):
        # BUG: Using same key!
        chain = random.normal(key, shape=(n_steps,))
        chains.append(chain)
    return jnp.array(chains)

# CORRECT: Different chains
def sample_correct(n_chains, n_steps):
    key = random.PRNGKey(0)
    chains = []
    for i in range(n_chains):
        key, subkey = random.split(key)  # Split key
        chain = random.normal(subkey, shape=(n_steps,))
        chains.append(chain)
    return jnp.array(chains)

wrong = sample_wrong(3, 100)
correct = sample_correct(3, 100)

print("Wrong approach (all identical):")
print(f"Chain 0 vs Chain 1 max diff: {jnp.max(jnp.abs(wrong[0] - wrong[1]))}")
print("✗ All chains are IDENTICAL\n")

print("Correct approach (different):")
print(f"Chain 0 vs Chain 1 max diff: {jnp.max(jnp.abs(correct[0] - correct[1]))}")
print("✓ All chains are DIFFERENT")

### Tricky Question 7.2: JIT Tracing Issues ⚫

**Common Problem**: JIT traces your function with specific shapes/types. Changing inputs can cause recompilation or errors.

**Question**: What's wrong here?

```python
@jit
def bad_function(x):
    n = x.shape[0]
    return jnp.sum(jnp.arange(n))  # Uses shape in traced graph
```

**Why it's bad**: `n` is traced as a constant. If you pass different shapes, it recompiles!

**Solution**: Avoid data-dependent control flow in JIT.

**Test below:**

In [ ]:
@jit
def problematic_func(x):
    """Uses shape in computation - causes recompilation for different shapes"""
    n = x.shape[0]
    return jnp.sum(jnp.arange(n) * x)

@jit
def better_func(x):
    """Uses shape but correctly - broadcasts instead of branching"""
    return jnp.sum(jnp.arange(len(x)) * x)

def test_jit_tracing():
    """Demonstrate JIT recompilation"""
    import warnings
    warnings.filterwarnings('ignore')
    
    print("Testing JIT with different input shapes:\n")
    
    x1 = jnp.array([1.0, 2.0, 3.0])  # shape (3,)
    x2 = jnp.array([1.0, 2.0, 3.0, 4.0])  # shape (4,)
    
    result1 = problematic_func(x1)
    result2 = problematic_func(x2)
    
    print(f"Shape (3,): {result1}")
    print(f"Shape (4,): {result2}")
    print("✓ Works, but recompiled for different shapes")
    print("(In real code, check XLA logs to see recompilation warnings)\n")
    
    # Better approach - works naturally with different shapes
    result1 = better_func(x1)
    result2 = better_func(x2)
    
    print(f"Better func shape (3,): {result1}")
    print(f"Better func shape (4,): {result2}")
    print("✓ No recompilation needed (broadcasting handles both)"

test_jit_tracing()

### Tricky Question 7.3: log-sum-exp Numerics ⚫

**Problem**: Computing log(Σ exp(x_i)) naively causes overflow/underflow.

**Question**: What's the stable way to compute this?

**Answer**: Log-sum-exp trick:
log(Σ exp(x_i)) = max(x) + log(Σ exp(x_i - max(x)))

**Why**: Subtracting the max prevents large exponents that overflow.

**Implement and verify below:**

In [ ]:
def logsumexp_naive(x):
    """
    NAIVE (WRONG): Direct computation - overflows!
    """
    return jnp.log(jnp.sum(jnp.exp(x)))

def logsumexp_stable(x):
    """
    STABLE (CORRECT): Use max subtraction trick
    """
    # FILL IN HERE
    # log(Σ exp(x_i)) = max(x) + log(Σ exp(x_i - max(x)))
    pass

# Test with large values
x_large = jnp.array([1000.0, 1001.0, 999.0])

print("Log-sum-exp stability test:")
print(f"Input: {x_large}\n")

# Naive approach
try:
    result_naive = logsumexp_naive(x_large)
    print(f"Naive result: {result_naive}")
except:
    print(f"Naive result: inf (OVERFLOW!) ✗\n")

# Stable approach
result_stable = logsumexp_stable(x_large)
print(f"Stable result: {result_stable} ✓")

# Verify correctness on small values
x_small = jnp.array([1.0, 2.0, 3.0])
result_naive_small = logsumexp_naive(x_small)
result_stable_small = logsumexp_stable(x_small)

print(f"\nVerification on small values:")
print(f"Naive:  {result_naive_small}")
print(f"Stable: {result_stable_small}")
print(f"Match: {'✓ YES' if jnp.allclose(result_naive_small, result_stable_small) else '✗ NO'}")

# JAX provides this built-in
print(f"\nJAX built-in scipy.special.logsumexp: {jax.scipy.special.logsumexp(x_large)}")

# Final Section: Interview Preparation Summary

## Key JAX Interview Topics (Priority Order)

### 🔥 Most Important:
1. **Immutability & Functional Updates** - `.at[]` syntax
2. **Automatic Differentiation** - `grad()`, chain rule, higher-order derivatives
3. **Vectorization** - `vmap()` to eliminate loops
4. **Random Number Generation** - Pure functional RNG, key splitting
5. **JIT Compilation** - When to use, tracing, recompilation

### Important:
6. **Linear Algebra** - Cholesky, eigenvalues, SVD
7. **Optimization** - Gradient descent, convergence
8. **Composing Transformations** - `jit(grad(vmap(...)))`
9. **Sampling Algorithms** - MCMC, HMC basics
10. **Numerical Stability** - Log-sum-exp, Cholesky for sampling

---

## Research Interview Format Overview

**Typical 1-hour interview structure:**
- 5 min: Introduction & question explanation
- 40-45 min: Coding (start easy, progressively harder)
- 5-10 min: Discussion of tradeoffs, optimization

**What interviewers are testing:**
✓ Can you write efficient JAX code?  
✓ Do you understand numerical computing fundamentals?  
✓ Can you optimize for both correctness and speed?  
✓ Do you know common pitfalls?  
✓ Can you reason about algorithm complexity?  

---

## Interview Strategy

### Before the Interview:
- [ ] Review this notebook thoroughly
- [ ] Make sure you can run all exercises without looking
- [ ] Practice explaining code out loud
- [ ] Know the JAX documentation structure

### During the Interview:
1. **Listen carefully** - Ask clarifying questions about constraints
2. **Think before coding** - Outline the algorithm first
3. **Start simple** - Get basic version working, then optimize
4. **Test on small inputs** - Verify correctness before scaling
5. **Explain your reasoning** - Interviewers want to understand your thought process
6. **Ask about requirements** - "Do I need to optimize for speed or memory?"

### Common Red Flags (avoid these):
✗ Modifying JAX arrays in place (they're immutable!)  
✗ Using same random key for multiple operations  
✗ Naive matrix inversion (use Cholesky instead)  
✗ Computing log(Σ exp(...)) directly (use logsumexp)  
✗ JIT-ing functions with data-dependent control flow  
✗ Forgetting to handle batch dimensions properly  

### Things to Emphasize:
✓ "I would test this on small inputs first"  
✓ "Let me verify gradients with numerical checking"  
✓ "This would benefit from JIT compilation"  
✓ "I can vectorize this with vmap instead of looping"  
✓ "For numerical stability, I'd use..."  

---

## Practice Checklist

**Exercises by difficulty level:**
- [ ] Section 1: Array operations (Basic)
- [ ] Section 2: Autodiff & optimization (Intermediate)  
- [ ] Section 3: Vectorization (Intermediate)
- [ ] Section 4: JIT performance (Intermediate)
- [ ] Section 5: Advanced algorithms (Advanced)
- [ ] Section 6: Research problems (Advanced)
- [ ] Section 7: Tricky questions (Advanced)

**Time estimate to master:**
- Basic exercises: 2-3 hours
- Intermediate exercises: 4-5 hours
- Advanced exercises: 6-8 hours
- **Total**: 12-16 hours

---

## Resources

**Official Documentation:**
- JAX docs: https://jax.readthedocs.io/
- JAX examples: https://jax.readthedocs.io/en/latest/notebooks/thinking_in_jax.html
- Flax neural networks: https://flax.readthedocs.io/

**Papers to understand:**
- JAX whitepaper concepts - AD, JIT, vmap
- MCMC basics for sampling questions
- How to Read Research Papers: Start with Abstract → Methodology → Results

**Common JAX Patterns:**
```python
# Pattern 1: Vectorized batch processing
batch_fn = vmap(single_fn, in_axes=(0, None))

# Pattern 2: Fast gradient computation
@jit
grad_fn = grad(loss_fn)

# Pattern 3: Random key management
key, subkey = random.split(key)
sample = random.normal(subkey, shape=...)

# Pattern 4: Array updates
arr_new = arr.at[indices].set(values)

# Pattern 5: Stable log computations
result = jax.scipy.special.logsumexp(x)
```

---

## Final Tips

1. **Practice writing code without looking at examples** - This is what you'll do in the interview
2. **Get comfortable with error messages** - JAX's errors are usually helpful
3. **Know your keyboard shortcuts** - Faster coding is better (but not at the cost of correctness)
4. **Have a "library" of algorithms ready** - Metropolis-Hastings, gradient descent, etc.
5. **Be honest about what you don't know** - Better than confidently giving wrong answers

---

## After This Notebook

**Next steps for mastery:**
1. Implement a full Hamiltonian Monte Carlo sampler
2. Build a variational autoencoder from scratch
3. Implement a neural ODE
4. Create a publication-ready research algorithm

**Good luck! 🚀**

## Appendix: Solution Hints & Common Patterns

### Array Operations
```python
# Update arrays (immutable)
arr.at[i].set(value)
arr.at[indices].set(values)
arr.at[i].add(value)

# Normalization
normalized = (x - jnp.min(x)) / (jnp.max(x) - jnp.min(x))
normalized = (x - jnp.mean(x)) / (jnp.std(x) + eps)

# Clipping
clipped = jnp.clip(x, min_val, max_val)
```

### Automatic Differentiation
```python
# Basic gradient
g = grad(f)(x)

# Higher-order
hess = grad(grad(f))(x)

# Multiple outputs
g = grad(f, argnums=(0, 1))

# Value and gradient
value, g = jax.value_and_grad(f)(x)
```

### Vectorization
```python
# Batch over first dimension
batch_fn = vmap(single_fn)

# Batch over specific axes
batch_fn = vmap(single_fn, in_axes=(0, None))

# Nested batching
nested_fn = vmap(vmap(single_fn))
```

### Random Numbers
```python
# Initialize key
key = random.PRNGKey(seed)

# Split for independence
key, subkey = random.split(key)

# Multiple keys
keys = random.split(key, n)

# Common distributions
random.normal(subkey, shape=(d,))
random.uniform(subkey, shape=(n,))
random.multivariate_normal(subkey, mean, cov)
```

### JIT Compilation
```python
# Decorate function
@jit
def fast_fn(x):
    return ...

# Or wrap explicitly
jitted_fn = jit(fn)

# With static arguments (don't recompile for these)
@jit
def my_fn(x, static_arg):
    return ...
```

### Linear Algebra
```python
# Cholesky decomposition
L = linalg.cholesky(A)
# A = L @ L.T

# Eigendecomposition  
evals, evecs = linalg.eigh(A)

# Solve linear system
x = linalg.solve(A, b)

# Determinant
det = linalg.det(A)
```

### Numerical Stability
```python
# Log-sum-exp
result = jax.scipy.special.logsumexp(x)  # Instead of log(sum(exp(x)))

# Gaussian log prob
log_prob = -0.5 * (d * jnp.log(2 * jnp.pi) + 
                   jnp.sum(jnp.log(cov_diag)) +
                   jnp.sum(((x - mean) / cov_std)**2))

# Matrix inversion
x = linalg.solve(A, b)  # Instead of A^-1 @ b
```

---

## Tips for Each Problem Type

### "Implement gradient descent"
- Write loss function that returns scalar
- Use `grad(loss)` to get gradient function
- Update: `x = x - lr * grad(x)`
- Compile with `@jit` for speed

### "Vectorize this operation"
- Write single-example function first
- Determine which axis to vectorize: `in_axes`
- Use `vmap(fn, in_axes=(0, None))`
- Compose with `@jit` for performance

### "Implement MCMC"
- Need: log probability function
- Need: proposal distribution
- Metropolis-Hastings: acceptance = min(1, exp(log_alpha))
- Use stop_gradient for observed data

### "Optimize this"
- Measure first with `time_it()`
- Apply `@jit` to hot path
- Use `vmap` to eliminate loops
- Check for numerical issues

### "Handle batch dimension"
- Determine batch axis size
- Use `vmap` over that axis
- Compose with `grad` if needing gradients
- Can nest `vmap` for multiple batch dims